# 01 — Exploratory Data Analysis: Miracle Sheets (Daasity / Snowflake)

**Project:** RetentionIQ — Predictive Customer Intelligence for Shopify Brands  
**Data Source:** Daasity Unified Order Schema (UOS) in Snowflake  
**Author:** [Your Name]  
**Date:** March 2026  

---

## Purpose

This notebook explores the Miracle Sheets dataset extracted by Daasity from Shopify into Snowflake.  
We answer three foundational questions before any modeling:

1. **What does the data look like?** — Volume, shape, completeness, and quality of orders and customers.
2. **How do customers behave?** — Purchase frequency, inter-purchase timing, revenue distribution, discount patterns.
3. **What is the right churn definition?** — Empirical analysis of repurchase cycles to define our churn window.

These findings directly inform feature engineering (notebook 02) and model design (notebooks 04–06).

---

## Table of Contents

1. [Setup & Connection](#1-setup)
2. [Schema Exploration](#2-schema)
3. [Orders Overview](#3-orders)
4. [Customer Overview](#4-customers)
5. [Purchase Frequency & Timing](#5-frequency)
6. [Revenue Distribution](#6-revenue)
7. [Discount & Refund Patterns](#7-discounts)
8. [Product Mix](#8-products)
9. [Churn Window Analysis](#9-churn-window)
10. [Key Findings & Next Steps](#10-findings)

---
## 1. Setup & Connection <a id='1-setup'></a>

We connect directly to the Snowflake warehouse where Daasity stores both raw Shopify data  
and its unified schemas (UOS, DRP, etc.).

**Prerequisites:**
- A `.env` file with Snowflake credentials (see `.env.example` in the repo root)
- `pip install snowflake-connector-python python-dotenv pandas matplotlib seaborn`

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import os
import warnings
from datetime import datetime

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
import snowflake.connector
from dotenv import load_dotenv

# ============================================================
# CONFIGURATION
# ============================================================
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

# Display settings for pandas
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print(f"Notebook run: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

In [ ]:
# ============================================================
# SNOWFLAKE CONNECTION
# ============================================================
# Load credentials from .env file (never hardcode secrets)
load_dotenv()

conn = snowflake.connector.connect(
    account=os.environ['SNOWFLAKE_ACCOUNT'],
    user=os.environ['SNOWFLAKE_USER'],
    password=os.environ['SNOWFLAKE_PASSWORD'],
    warehouse=os.environ['SNOWFLAKE_WAREHOUSE'],
    database=os.environ['SNOWFLAKE_DATABASE'],
    role=os.environ.get('SNOWFLAKE_ROLE', 'SYSADMIN'),
)

def query(sql: str) -> pd.DataFrame:
    """Execute a SQL query against Snowflake and return a pandas DataFrame.
    
    Uses the existing connection `conn`. Column names are lowercased for
    consistency across Snowflake (which uppercases by default) and Python.
    """
    df = pd.read_sql(sql, conn)
    df.columns = [c.lower() for c in df.columns]
    return df

# Quick connectivity test
test = query("SELECT CURRENT_TIMESTAMP() AS ts, CURRENT_DATABASE() AS db, CURRENT_WAREHOUSE() AS wh")
print(f"Connected to: {test['db'].iloc[0]} | Warehouse: {test['wh'].iloc[0]} | Time: {test['ts'].iloc[0]}")

---
## 2. Schema Exploration <a id='2-schema'></a>

Before querying data, let's understand what schemas and tables Daasity has created.  
We expect to find:
- **Integration schemas** (raw Shopify data): `shopify.*`
- **Unified schemas** (modeled): `uos.*`, `ums.*`, `uns.*`, `uts.*`
- **Data marts** (reporting): `drp.*`, `dm_mkt.*`

**⚠️ Adapt the database/schema names below to match your actual Snowflake setup.**

In [ ]:
# ============================================================
# LIST ALL SCHEMAS IN THE DATABASE
# ============================================================
schemas = query("SHOW SCHEMAS")
print(f"Found {len(schemas)} schemas:")
schemas[['name', 'database_name']].head(20)

In [ ]:
# ============================================================
# LIST TABLES IN THE UOS SCHEMA
# This is our primary data source for feature engineering.
# ============================================================
# NOTE: Adjust schema name if different in your Snowflake instance
UOS_SCHEMA = 'uos'  # <-- CHANGE THIS if your UOS schema has a different name

uos_tables = query(f"SHOW TABLES IN SCHEMA {UOS_SCHEMA}")
print(f"\nUOS Schema — {len(uos_tables)} tables:")
uos_tables[['name', 'rows', 'bytes']].sort_values('name')

In [ ]:
# ============================================================
# INSPECT KEY TABLE COLUMNS
# Understanding column names and types before writing queries.
# ============================================================
for table_name in ['customers', 'orders', 'order_line_items', 'product_variants', 'refunds']:
    cols = query(f"DESCRIBE TABLE {UOS_SCHEMA}.{table_name}")
    print(f"\n{'='*60}")
    print(f"  {UOS_SCHEMA}.{table_name} — {len(cols)} columns")
    print(f"{'='*60}")
    display(cols[['name', 'type']].head(25))

---
## 3. Orders Overview <a id='3-orders'></a>

High-level stats on the order table: total volume, date range, financial status breakdown.  
This tells us the scale of data we're working with.

In [ ]:
# ============================================================
# ORDERS: HIGH-LEVEL SUMMARY
# ============================================================
# NOTE: Adjust column names below if your UOS uses different naming conventions.
# Common Daasity UOS columns: order_id, customer_id, order_date, total_price,
# financial_status, discount_code, discount_amount, source_system

orders_summary = query(f"""
    SELECT
        COUNT(*)                          AS total_orders,
        COUNT(DISTINCT customer_id)       AS unique_customers,
        MIN(order_date)                   AS earliest_order,
        MAX(order_date)                   AS latest_order,
        ROUND(SUM(total_price), 2)        AS total_revenue,
        ROUND(AVG(total_price), 2)        AS avg_order_value,
        ROUND(MEDIAN(total_price), 2)     AS median_order_value
    FROM {UOS_SCHEMA}.orders
""")

print("ORDERS — HIGH-LEVEL SUMMARY")
print("=" * 50)
for col in orders_summary.columns:
    val = orders_summary[col].iloc[0]
    print(f"  {col:<25} {val}")

In [ ]:
# ============================================================
# ORDERS OVER TIME — Monthly trend
# Are we seeing growth? Seasonality? Decline?
# ============================================================
orders_monthly = query(f"""
    SELECT
        DATE_TRUNC('month', order_date)   AS month,
        COUNT(*)                           AS order_count,
        COUNT(DISTINCT customer_id)        AS unique_customers,
        ROUND(SUM(total_price), 2)         AS revenue
    FROM {UOS_SCHEMA}.orders
    GROUP BY 1
    ORDER BY 1
""")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Orders per month
axes[0].bar(orders_monthly['month'], orders_monthly['order_count'], color='#534AB7', alpha=0.8, width=20)
axes[0].set_title('Monthly Order Volume', fontweight='bold')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Orders')
axes[0].tick_params(axis='x', rotation=45)

# Revenue per month
axes[1].bar(orders_monthly['month'], orders_monthly['revenue'], color='#0F6E56', alpha=0.8, width=20)
axes[1].set_title('Monthly Revenue', fontweight='bold')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Revenue ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# FINANCIAL STATUS BREAKDOWN
# How many orders are paid, refunded, partially refunded, etc.?
# This informs which orders to include in our feature engineering.
# ============================================================
fin_status = query(f"""
    SELECT
        financial_status,
        COUNT(*)            AS order_count,
        ROUND(SUM(total_price), 2) AS total_revenue
    FROM {UOS_SCHEMA}.orders
    GROUP BY 1
    ORDER BY 2 DESC
""")

fin_status['pct'] = (fin_status['order_count'] / fin_status['order_count'].sum() * 100).round(1)
print("FINANCIAL STATUS BREAKDOWN")
display(fin_status)

---
## 4. Customer Overview <a id='4-customers'></a>

Understanding the customer base: how many are repeat vs. one-time buyers,  
the distribution of order counts, and customer acquisition over time.

In [ ]:
# ============================================================
# ORDERS PER CUSTOMER — The most important distribution
# Most DTC brands see 60-80% single-order customers.
# ============================================================
orders_per_customer = query(f"""
    SELECT
        customer_id,
        COUNT(*)                      AS order_count,
        SUM(total_price)              AS total_spent,
        MIN(order_date)               AS first_order,
        MAX(order_date)               AS last_order,
        DATEDIFF('day', MIN(order_date), MAX(order_date)) AS tenure_days
    FROM {UOS_SCHEMA}.orders
    WHERE financial_status NOT IN ('refunded', 'voided')
    GROUP BY customer_id
""")

total_customers = len(orders_per_customer)
one_time = (orders_per_customer['order_count'] == 1).sum()
repeat = total_customers - one_time

print(f"Total customers:    {total_customers:,}")
print(f"One-time buyers:    {one_time:,} ({one_time/total_customers*100:.1f}%)")
print(f"Repeat buyers:      {repeat:,} ({repeat/total_customers*100:.1f}%)")
print(f"\nOrders per customer:")
print(orders_per_customer['order_count'].describe().to_string())

In [ ]:
# ============================================================
# ORDER COUNT DISTRIBUTION — Histogram
# We cap at 10+ orders for readability (the tail is very long).
# ============================================================
order_counts = orders_per_customer['order_count'].clip(upper=10)

fig, ax = plt.subplots(figsize=(10, 5))
counts = order_counts.value_counts().sort_index()
bars = ax.bar(counts.index, counts.values, color='#534AB7', alpha=0.85, edgecolor='white')

# Add percentage labels on top of each bar
for bar, val in zip(bars, counts.values):
    pct = val / total_customers * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + total_customers*0.005,
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xlabel('Number of Orders')
ax.set_ylabel('Number of Customers')
ax.set_title('Distribution of Orders per Customer', fontweight='bold')
ax.set_xticks(range(1, 11))
ax.set_xticklabels([str(i) if i < 10 else '10+' for i in range(1, 11)])
plt.tight_layout()
plt.show()

print(f"\n→ Insight: {one_time/total_customers*100:.0f}% of customers buy only once. "
      f"Converting even 10% of these to repeat buyers could significantly impact revenue.")

---
## 5. Purchase Frequency & Timing <a id='5-frequency'></a>

This is **critical** for two reasons:
1. The **inter-purchase time distribution** defines our churn window (Section 9).
2. The **BG/NBD model** (for CLV and next-purchase timing) is built on frequency and recency.

We analyze only repeat customers here (customers with 2+ orders).

In [ ]:
# ============================================================
# INTER-PURCHASE TIME — Days between consecutive orders
# This is the empirical basis for defining "churn."
# ============================================================
inter_purchase = query(f"""
    WITH ordered AS (
        SELECT
            customer_id,
            order_date,
            LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS prev_order_date
        FROM {UOS_SCHEMA}.orders
        WHERE financial_status NOT IN ('refunded', 'voided')
    )
    SELECT
        customer_id,
        DATEDIFF('day', prev_order_date, order_date) AS days_between_orders
    FROM ordered
    WHERE prev_order_date IS NOT NULL
      AND DATEDIFF('day', prev_order_date, order_date) > 0
""")

print(f"Total inter-purchase intervals: {len(inter_purchase):,}")
print(f"\nDays between orders — Summary Statistics:")
print(inter_purchase['days_between_orders'].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]
).to_string())

In [ ]:
# ============================================================
# INTER-PURCHASE TIME DISTRIBUTION — Histogram + Key Percentiles
# The 90th and 95th percentiles are candidates for the churn window.
# ============================================================
days = inter_purchase['days_between_orders']
p50 = days.quantile(0.50)
p75 = days.quantile(0.75)
p90 = days.quantile(0.90)
p95 = days.quantile(0.95)

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(days.clip(upper=365), bins=60, color='#0F6E56', alpha=0.8, edgecolor='white')

# Mark key percentiles
for pval, plabel, color in [(p50, 'P50', '#534AB7'), (p75, 'P75', '#D85A30'), 
                              (p90, 'P90', '#E24B4A'), (p95, 'P95', '#791F1F')]:
    ax.axvline(pval, color=color, linestyle='--', linewidth=2, alpha=0.8)
    ax.text(pval + 3, ax.get_ylim()[1] * 0.85, f'{plabel}: {pval:.0f}d', 
            color=color, fontweight='bold', fontsize=10)

ax.set_xlabel('Days Between Consecutive Orders')
ax.set_ylabel('Frequency')
ax.set_title('Inter-Purchase Time Distribution (capped at 365 days)', fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n→ Key finding: Median repurchase time is {p50:.0f} days.")
print(f"  90th percentile is {p90:.0f} days — this is a strong churn window candidate.")
print(f"  95th percentile is {p95:.0f} days — more conservative churn window.")

---
## 6. Revenue Distribution <a id='6-revenue'></a>

How is revenue distributed across customers?  
DTC brands typically follow a Pareto pattern: ~20% of customers drive ~80% of revenue.

In [ ]:
# ============================================================
# REVENUE PER CUSTOMER — Distribution
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of total spend per customer (capped at reasonable value)
spend = orders_per_customer['total_spent']
cap = spend.quantile(0.99)  # Cap at 99th percentile for visualization
axes[0].hist(spend.clip(upper=cap), bins=50, color='#D85A30', alpha=0.8, edgecolor='white')
axes[0].set_xlabel('Total Customer Spend ($)')
axes[0].set_ylabel('Number of Customers')
axes[0].set_title('Revenue per Customer Distribution', fontweight='bold')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Pareto curve: cumulative revenue share
sorted_spend = spend.sort_values(ascending=False).reset_index(drop=True)
cum_pct = sorted_spend.cumsum() / sorted_spend.sum() * 100
cust_pct = np.arange(1, len(cum_pct) + 1) / len(cum_pct) * 100

axes[1].plot(cust_pct, cum_pct, color='#534AB7', linewidth=2)
axes[1].axhline(80, color='#E24B4A', linestyle='--', alpha=0.6)
axes[1].axvline(20, color='#E24B4A', linestyle='--', alpha=0.6)
axes[1].fill_between(cust_pct, cum_pct, alpha=0.1, color='#534AB7')
axes[1].set_xlabel('% of Customers (sorted by spend, descending)')
axes[1].set_ylabel('% of Cumulative Revenue')
axes[1].set_title('Revenue Concentration (Pareto Curve)', fontweight='bold')

# Calculate actual Pareto ratio
top_20_rev = sorted_spend.iloc[:int(len(sorted_spend) * 0.20)].sum()
pareto_pct = top_20_rev / sorted_spend.sum() * 100
axes[1].text(25, 70, f'Top 20% = {pareto_pct:.0f}% of revenue', fontsize=11, fontweight='bold', color='#E24B4A')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# AOV (Average Order Value) DISTRIBUTION
# ============================================================
aov = query(f"""
    SELECT total_price
    FROM {UOS_SCHEMA}.orders
    WHERE financial_status NOT IN ('refunded', 'voided')
      AND total_price > 0
""")

fig, ax = plt.subplots(figsize=(10, 5))
aov_cap = aov['total_price'].clip(upper=aov['total_price'].quantile(0.99))
ax.hist(aov_cap, bins=50, color='#0F6E56', alpha=0.8, edgecolor='white')
ax.axvline(aov['total_price'].median(), color='#534AB7', linestyle='--', linewidth=2)
ax.text(aov['total_price'].median() + 5, ax.get_ylim()[1]*0.8, 
        f"Median: ${aov['total_price'].median():.0f}", color='#534AB7', fontweight='bold')
ax.set_xlabel('Order Value ($)')
ax.set_ylabel('Frequency')
ax.set_title('Average Order Value Distribution', fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

---
## 7. Discount & Refund Patterns <a id='7-discounts'></a>

Discount dependency and refund behavior are strong churn predictors.  
Customers who only buy with discounts may have lower long-term value.

In [ ]:
# ============================================================
# DISCOUNT USAGE
# ============================================================
discount_stats = query(f"""
    SELECT
        CASE WHEN discount_code IS NOT NULL AND discount_code != '' THEN 'With Discount' 
             ELSE 'Full Price' END AS order_type,
        COUNT(*)                    AS order_count,
        ROUND(AVG(total_price), 2)  AS avg_order_value,
        ROUND(SUM(total_price), 2)  AS total_revenue
    FROM {UOS_SCHEMA}.orders
    WHERE financial_status NOT IN ('refunded', 'voided')
    GROUP BY 1
""")

print("DISCOUNT vs FULL PRICE ORDERS")
display(discount_stats)

# Per-customer discount dependency
cust_discount = query(f"""
    SELECT
        customer_id,
        COUNT(*) AS total_orders,
        SUM(CASE WHEN discount_code IS NOT NULL AND discount_code != '' THEN 1 ELSE 0 END) AS discount_orders
    FROM {UOS_SCHEMA}.orders
    WHERE financial_status NOT IN ('refunded', 'voided')
    GROUP BY customer_id
    HAVING COUNT(*) >= 2
""")

cust_discount['discount_rate'] = cust_discount['discount_orders'] / cust_discount['total_orders']
always_discount = (cust_discount['discount_rate'] == 1.0).sum()
never_discount = (cust_discount['discount_rate'] == 0.0).sum()

print(f"\nAmong repeat customers (2+ orders):")
print(f"  Always use discount: {always_discount} ({always_discount/len(cust_discount)*100:.1f}%)")
print(f"  Never use discount:  {never_discount} ({never_discount/len(cust_discount)*100:.1f}%)")

In [ ]:
# ============================================================
# REFUND ANALYSIS
# ============================================================
refund_stats = query(f"""
    SELECT
        COUNT(DISTINCT r.customer_id) AS customers_with_refunds,
        COUNT(DISTINCT r.refund_id)   AS total_refunds,
        ROUND(SUM(r.refund_amount), 2) AS total_refund_amount
    FROM {UOS_SCHEMA}.refunds r
""")

print("REFUND SUMMARY")
print(f"  Customers with refunds: {refund_stats['customers_with_refunds'].iloc[0]:,}")
print(f"  Total refunds:          {refund_stats['total_refunds'].iloc[0]:,}")
print(f"  Total refund amount:    ${refund_stats['total_refund_amount'].iloc[0]:,.2f}")
print(f"  Refund rate:            {refund_stats['customers_with_refunds'].iloc[0] / total_customers * 100:.1f}% of customers")

---
## 8. Product Mix <a id='8-products'></a>

What products do customers buy? Which product types correlate with repeat purchases?  
This feeds into the product affinity features for our models.

In [ ]:
# ============================================================
# TOP PRODUCT TYPES BY ORDER VOLUME
# ============================================================
product_mix = query(f"""
    SELECT
        pv.product_type,
        COUNT(DISTINCT o.order_id)      AS order_count,
        COUNT(DISTINCT o.customer_id)   AS customer_count,
        ROUND(SUM(li.price * li.quantity), 2) AS revenue
    FROM {UOS_SCHEMA}.order_line_items li
    JOIN {UOS_SCHEMA}.orders o ON li.order_id = o.order_id
    JOIN {UOS_SCHEMA}.product_variants pv ON li.product_variant_id = pv.product_variant_id
    WHERE o.financial_status NOT IN ('refunded', 'voided')
    GROUP BY 1
    ORDER BY 4 DESC
    LIMIT 15
""")

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(product_mix['product_type'][::-1], product_mix['revenue'][::-1], 
               color='#534AB7', alpha=0.85)
ax.set_xlabel('Revenue ($)')
ax.set_title('Top Product Types by Revenue', fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

---
## 9. Churn Window Analysis <a id='9-churn-window'></a>

**This is the single most important analysis in the notebook.**

Defining "churn" for a non-contractual business like Miracle Sheets requires choosing a time  
window after which a customer who hasn't purchased is considered churned. There is no universal  
standard — it must be calibrated to the brand's actual repurchase behavior.

**Approach:** We use the inter-purchase time distribution from Section 5 and compare different  
window candidates (90, 120, 150, 180 days) to see what fraction of "active" customers would  
be captured by each definition.

In [ ]:
# ============================================================
# CHURN WINDOW SENSITIVITY ANALYSIS
# For each candidate window, calculate:
# - What % of repeat customers would be labeled as "churned"
# - What % of inter-purchase intervals fall within the window
# ============================================================
windows = [60, 90, 120, 150, 180, 240, 365]
days_col = inter_purchase['days_between_orders']

print("CHURN WINDOW SENSITIVITY ANALYSIS")
print("=" * 65)
print(f"{'Window (days)':<18} {'% intervals within':<22} {'Interpretation'}")
print("-" * 65)
for w in windows:
    pct_within = (days_col <= w).mean() * 100
    print(f"{w:<18} {pct_within:>6.1f}%               "
          f"{'← captures most repurchases' if pct_within > 85 else '← may miss slow buyers' if pct_within < 75 else ''}")

print(f"\n→ Recommendation: Use the window where ~90% of inter-purchase intervals are captured.")
print(f"  This avoids labeling slow-but-active customers as churned.")

In [ ]:
# ============================================================
# SURVIVAL CURVE — What fraction of customers repurchase by day N?
# ============================================================
max_days = 365
survival_data = []
for d in range(1, max_days + 1):
    pct_repurchased = (days_col <= d).mean() * 100
    survival_data.append({'day': d, 'pct_repurchased': pct_repurchased})

survival_df = pd.DataFrame(survival_data)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(survival_df['day'], survival_df['pct_repurchased'], color='#534AB7', linewidth=2)
ax.fill_between(survival_df['day'], survival_df['pct_repurchased'], alpha=0.1, color='#534AB7')

# Mark candidate windows
for w, color in [(90, '#D85A30'), (120, '#E24B4A'), (180, '#791F1F')]:
    pct = (days_col <= w).mean() * 100
    ax.axvline(w, color=color, linestyle='--', alpha=0.7)
    ax.plot(w, pct, 'o', color=color, markersize=8)
    ax.text(w + 5, pct - 3, f'{w}d: {pct:.0f}%', color=color, fontweight='bold', fontsize=10)

ax.set_xlabel('Days Since Last Order')
ax.set_ylabel('% of Repeat Customers Who Repurchased by This Day')
ax.set_title('Repurchase Survival Curve — Churn Window Selection', fontweight='bold')
ax.set_xlim(0, max_days)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

---
## 10. Key Findings & Next Steps <a id='10-findings'></a>

### Summary of Findings

| Finding | Value | Implication |
|---------|-------|-------------|
| Total customers | *fill from output* | Scale of the dataset |
| One-time buyer rate | *fill from output* | High churn potential — core problem to solve |
| Median inter-purchase time | *fill from output* | Informs BG/NBD and churn window |
| P90 inter-purchase time | *fill from output* | Recommended churn window |
| Top 20% customer revenue share | *fill from output* | Revenue concentration — VIP retention is critical |
| Discount dependency rate | *fill from output* | Feature for churn model |
| Refund rate | *fill from output* | Feature for churn model |

### Decisions Made

- **Churn window:** [To be filled based on the analysis above — likely 120–180 days for sheets]
- **Orders to include:** Exclude `financial_status IN ('refunded', 'voided')` for feature engineering
- **Feature engineering focus:** RFM, discount behavior, refund rate, product type diversity

### Next Steps

1. **Notebook 02** — Build the `customer_features` SQL and validate in Python
2. **Notebook 04** — Train BG/NBD + Gamma-Gamma for CLV prediction
3. **Notebook 05** — Train churn classifiers using the chosen churn window

In [ ]:
# ============================================================
# CLEANUP — Close Snowflake connection
# ============================================================
conn.close()
print("Snowflake connection closed.")